# ArmanNN Training on Google Colab

This notebook trains ArmanNN on FineWeb-Edu using an A100 GPU.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **A100 GPU**
2. Run all cells in order

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/mhd-rahman/Arman-NN.git
%cd Arman-NN

In [ ]:
!pip install -q torch datasets transformers numpy flash-attn --no-build-isolation

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load the dataset

In [ ]:
import sys
sys.path.insert(0, ".")

import random
import torch

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset


# ============================================================
# CONFIG
# ============================================================

SEQ_LEN = 1024
SEED = 42

# Set this if StarCoderData or another gated dataset requires it.
# Prefer environment variables in real training instead of hardcoding.
HF_TOKEN = None

tokenizer = AutoTokenizer.from_pretrained("gpt2")


# ============================================================
# PRETRAINING DATA MIX
# ============================================================
#
# Target ~12B tokens:
#
# FineWeb-Edu     7.5B  = 62.5%
# Code            1.5B  = 12.5%
# Wikipedia       1.0B  =  8.3%
# OpenWebMath     2.0B  = 16.7%
#
# These weights control sampling probability. They DO NOT
# require downloading 12B tokens ahead of time.
# ============================================================

DATASET_WEIGHTS = {
    "fineweb": 0.625,
    "code": 0.125,
    "wikipedia": 0.083,
    "math": 0.167,
}


# ============================================================
# CODE LANGUAGE MIX
# ============================================================
#
# Internal mixture INSIDE the 12.5% code allocation.
#
# This means, for example:
#
# Python:
#     12.5% * 26.7% ≈ 3.34% of total training tokens
#
# ============================================================

CODE_LANGUAGES = {
    "python": 0.267,
    "javascript": 0.120,
    "typescript": 0.100,
    "cpp": 0.087,
    "java": 0.087,
    "c": 0.073,
    "rust": 0.067,
    "go": 0.067,
    "shell": 0.033,
    "sql": 0.033,
    "html": 0.020,
    "css": 0.013,
    "json": 0.013,
    "yaml": 0.010,
    "markdown": 0.010,
}


# Normalize in case floating point values don't sum to exactly 1.
total_code_weight = sum(CODE_LANGUAGES.values())

CODE_LANGUAGES = {
    language: weight / total_code_weight
    for language, weight in CODE_LANGUAGES.items()
}


# ============================================================
# HELPER
# ============================================================

def stream_hf_dataset(
    dataset_name,
    subset=None,
    split="train",
    text_field="text",
    data_dir=None,
    token=HF_TOKEN,
    seed=SEED,
    skip=0,
):
    """
    Creates an infinite-ish text generator from a Hugging Face
    streaming dataset.

    skip:
        Useful for creating a validation stream that doesn't overlap
        with the beginning of the training stream.
    """

    kwargs = {
        "path": dataset_name,
        "split": split,
        "streaming": True,
        "token": token,
    }

    if subset is not None:
        kwargs["name"] = subset

    if data_dir is not None:
        kwargs["data_dir"] = data_dir

    ds = load_dataset(**kwargs)

    ds = ds.shuffle(
        seed=seed,
        buffer_size=10_000,
    )

    if skip > 0:
        ds = ds.skip(skip)

    for example in ds:

        text = example.get(text_field)

        if text is None:
            continue

        if not isinstance(text, str):
            continue

        if not text.strip():
            continue

        yield text


# ============================================================
# FINEWEB-EDU
# ============================================================

def fineweb_stream(seed=SEED, skip=0):

    yield from stream_hf_dataset(
        dataset_name="HuggingFaceFW/fineweb-edu",
        subset="sample-10BT",
        split="train",
        text_field="text",
        seed=seed,
        skip=skip,
    )


# ============================================================
# WIKIPEDIA
# ============================================================

def wikipedia_stream(seed=SEED, skip=0):

    yield from stream_hf_dataset(
        dataset_name="wikimedia/wikipedia",
        subset="20231101.en",
        split="train",
        text_field="text",
        seed=seed,
        skip=skip,
    )


# ============================================================
# OPEN WEB MATH
# ============================================================

def math_stream(seed=SEED, skip=0):

    yield from stream_hf_dataset(
        dataset_name="open-web-math/open-web-math",
        split="train",
        text_field="text",
        seed=seed,
        skip=skip,
    )


# ============================================================
# STARCODER DATA
# ============================================================

def single_code_language_stream(
    language,
    seed=SEED,
    skip=0,
):

    """
    StarCoderData separates languages through data_dir.

    Its source-code field is `content`, not `text`.
    """

    yield from stream_hf_dataset(
        dataset_name="bigcode/starcoderdata",
        split="train",
        text_field="content",
        data_dir=language,
        token=HF_TOKEN,
        seed=seed,
        skip=skip,
    )


def code_stream(seed=SEED, skip=0):

    """
    Produces code according to our internal language distribution.
    """

    languages = list(CODE_LANGUAGES.keys())

    weights = [
        CODE_LANGUAGES[language]
        for language in languages
    ]

    streams = {
        language: iter(
            single_code_language_stream(
                language,
                seed=seed + i,
                skip=skip,
            )
        )
        for i, language in enumerate(languages)
    }

    rng = random.Random(seed)

    while True:

        language = rng.choices(
            languages,
            weights=weights,
            k=1,
        )[0]

        try:

            yield next(streams[language])

        except StopIteration:

            # Restart that particular language stream if exhausted.

            streams[language] = iter(
                single_code_language_stream(
                    language,
                    seed=seed + 1000 + languages.index(language),
                    skip=skip,
                )
            )


# ============================================================
# COMBINED PRETRAINING STREAM
# ============================================================

class MixedPretrainingDataset(IterableDataset):

    def __init__(
        self,
        tokenizer,
        seq_len,
        seed=SEED,
        validation=False,
        skip_tokens=0,
    ):
        super().__init__()

        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.seed = seed
        self.validation = validation
        self.skip_tokens = skip_tokens


    def __iter__(self):

        worker = torch.utils.data.get_worker_info()

        if worker is None:
            worker_id = 0
        else:
            worker_id = worker.id

        seed = self.seed + worker_id

        # Use different positions/seeds for validation.
        #
        # This is intentionally large so eval isn't just taking
        # the same documents training sees first.
        if self.validation:

            skip = 50_000
            seed += 10_000

        else:

            skip = 0

        streams = {

            "fineweb": iter(
                fineweb_stream(
                    seed=seed + 1,
                    skip=skip,
                )
            ),

            "code": iter(
                code_stream(
                    seed=seed + 2,
                    skip=skip,
                )
            ),

            "wikipedia": iter(
                wikipedia_stream(
                    seed=seed + 3,
                    skip=skip,
                )
            ),

            "math": iter(
                math_stream(
                    seed=seed + 4,
                    skip=skip,
                )
            ),
        }

        names = list(DATASET_WEIGHTS.keys())

        weights = [
            DATASET_WEIGHTS[name]
            for name in names
        ]

        rng = random.Random(seed)

        token_buffer = []

        # Skip tokens for resume (fast-forward through already-seen data)
        tokens_skipped = 0
        skip_target = self.skip_tokens

        while True:

            # Select which dataset the next document comes from.
            source = rng.choices(
                names,
                weights=weights,
                k=1,
            )[0]

            try:

                text = next(streams[source])

            except StopIteration:

                # If a stream runs out, simply rebuild it.
                if source == "fineweb":

                    streams[source] = iter(
                        fineweb_stream(seed=seed + 100)
                    )

                elif source == "code":

                    streams[source] = iter(
                        code_stream(seed=seed + 200)
                    )

                elif source == "wikipedia":

                    streams[source] = iter(
                        wikipedia_stream(seed=seed + 300)
                    )

                elif source == "math":

                    streams[source] = iter(
                        math_stream(seed=seed + 400)
                    )

                continue

            # --------------------------------------------------
            # Tokenize document
            # --------------------------------------------------

            tokens = self.tokenizer.encode(
                text,
                add_special_tokens=False,
            )

            if len(tokens) == 0:
                continue

            # Skip tokens for resume
            if tokens_skipped < skip_target:
                tokens_skipped += len(tokens)
                continue

            # Add an EOS separator between documents.
            if self.tokenizer.eos_token_id is not None:

                tokens.append(
                    self.tokenizer.eos_token_id
                )

            token_buffer.extend(tokens)

            # --------------------------------------------------
            # Produce fixed-size LM sequences
            # --------------------------------------------------

            while len(token_buffer) >= self.seq_len + 1:

                chunk = token_buffer[
                    : self.seq_len + 1
                ]

                # IMPORTANT:
                #
                # consume SEQ_LEN tokens rather than SEQ_LEN + 1
                # because token N becomes the first context token
                # of the next sequence.
                token_buffer = token_buffer[
                    self.seq_len :
                ]

                x = torch.tensor(
                    chunk[:-1],
                    dtype=torch.long,
                )

                y = torch.tensor(
                    chunk[1:],
                    dtype=torch.long,
                )

                yield x, y


# ============================================================
# TRAIN STREAM
# ============================================================

train_dataset = MixedPretrainingDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
    seed=42,
    validation=False,
)


# ============================================================
# VALIDATION — held-out benchmark + mixed domain eval
# ============================================================
# Builds eval once and caches to disk. Subsequent runs load instantly.

EVAL_CACHE_PATH = "eval_dataset.pt"

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

if os.path.exists(EVAL_CACHE_PATH):
    print(f"Loading cached eval dataset from {EVAL_CACHE_PATH}...")
    all_eval_samples = torch.load(EVAL_CACHE_PATH, weights_only=False)
    eval_dataset = ListDataset(all_eval_samples)
    print(f"Eval: {len(eval_dataset):,} sequences (loaded from cache)")
else:
    print("Building eval dataset (first time only, will cache to disk)...")

    # 1. Wikitext-2 test set (fixed benchmark)
    print("  Loading Wikitext-2 test...")
    eval_ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    wikitext_tokens = []
    for example in eval_ds:
        text = example["text"]
        if text.strip():
            wikitext_tokens.extend(tokenizer.encode(text, add_special_tokens=False))

    wikitext_samples = []
    for i in range(0, len(wikitext_tokens) - SEQ_LEN, SEQ_LEN):
        x = torch.tensor(wikitext_tokens[i:i + SEQ_LEN], dtype=torch.long)
        y = torch.tensor(wikitext_tokens[i + 1:i + SEQ_LEN + 1], dtype=torch.long)
        wikitext_samples.append((x, y))
    del wikitext_tokens, eval_ds
    print(f"    Wikitext-2: {len(wikitext_samples)} sequences")

    # 2. Mixed domain eval (randomized)
    print("  Loading mixed domain eval...")
    mixed_eval_stream = MixedPretrainingDataset(
        tokenizer=tokenizer,
        seq_len=SEQ_LEN,
        seed=random.randint(0, 100000),
        validation=True,
    )
    mixed_eval_samples = []
    for i, (x, y) in enumerate(mixed_eval_stream):
        mixed_eval_samples.append((x, y))
        if i >= 2499:
            break
    del mixed_eval_stream
    print(f"    Mixed domain: {len(mixed_eval_samples)} sequences")

    # Combine and save
    all_eval_samples = wikitext_samples + mixed_eval_samples
    del wikitext_samples, mixed_eval_samples
    import gc; gc.collect()

    torch.save(all_eval_samples, EVAL_CACHE_PATH)
    print(f"  Saved eval cache to {EVAL_CACHE_PATH}")

    eval_dataset = ListDataset(all_eval_samples)
    print(f"\nTotal eval: {len(eval_dataset):,} sequences (wikitext + mixed domain)")

print("Mixed streaming train dataset ready")

print("\nTraining mixture:")

for name, weight in DATASET_WEIGHTS.items():
    print(
        f"  {name:<12} {weight * 100:>5.1f}%"
    )

## 3. Configure and build the model

In [ ]:
from arman.model import ArmanConfig, ArmanNN

config = ArmanConfig(
    vocab_size=50257,       # GPT-2 tokenizer vocab
    d_model=1024,
    n_layers=12,
    n_heads=16,
    max_seq_len=1024,
    mlp_hidden=2816,
    expert_hidden=1792,
    n_experts=4,
    moe_top_k=2,
    ssm_state_size=128,
    use_graph=False,        # Disabled — not feeding graph data during training
    use_memory=False,       # Disabled — adds noise without structured input
)

model = ArmanNN(config)
print(f"Parameters: {model.parameter_count():,}")

## 4. Train with the Trainer

In [ ]:
import logging
import time
from pathlib import Path

# Clear any stale GPU memory from previous runs
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Force logging output to display in notebook cells
log = logging.getLogger()
log.setLevel(logging.INFO)
log.handlers = []
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
log.addHandler(handler)

from arman.training.scheduler import get_cosine_schedule_with_warmup
from arman.training.checkpointing import save_checkpoint, load_checkpoint, find_latest_checkpoint
from arman.training.evaluator import Evaluator, EvalConfig
from torch.utils.data import DataLoader

# Training hyperparameters (H200 141GB)
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
BATCH_SIZE = 32
GRAD_ACCUM = 1        # Effective batch = 32 sequences × 1024 tokens = ~32k tokens/step
WARMUP_STEPS = 0
TOTAL_STEPS = 76000
MIN_LR_RATIO = 0.1
SAVE_EVERY = 2000
EVAL_EVERY = 2000
LOG_EVERY = 50
CHECKPOINT_DIR = "checkpoints"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.enable_gradient_checkpointing()  # Saves ~60% memory, ~30% slower
print(f"Device: {device}")
print(f"Gradient checkpointing: disabled")

# Optimizer with weight decay separation
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim >= 2 and "norm" not in n]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and (p.ndim < 2 or "norm" in n)]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95))

# Resume from checkpoint if available
global_step = 0
ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if ckpt:
    info = load_checkpoint(ckpt, model, optimizer=optimizer, scheduler=None, device="cpu", strict=False)
    global_step = info["step"]
    print(f"Resumed from {ckpt} at step {global_step}")

scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, TOTAL_STEPS, MIN_LR_RATIO)
# Fast-forward scheduler to current step on resume
for _ in range(global_step):
    scheduler.step()

# bf16 — no GradScaler needed (more stable than fp16)
AMP_DTYPE = torch.bfloat16

# DataLoader for streaming dataset — skip tokens already seen on resume
tokens_seen = global_step * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN
if tokens_seen > 0:
    print(f"Skipping ~{tokens_seen:,} tokens to resume from step {global_step}...")
train_dataset = MixedPretrainingDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
    seed=42,
    validation=False,
    skip_tokens=tokens_seen,
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

# Training loop
model.train()
optimizer.zero_grad(set_to_none=True)
data_iter = iter(train_loader)
step_start = time.time()

print(f"Training for {TOTAL_STEPS} steps | effective batch = {BATCH_SIZE * GRAD_ACCUM} | tokens/step = {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN:,}")
print(f"Starting from step {global_step}")

while global_step < TOTAL_STEPS:
    accum_loss = 0.0
    accum_aux = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            out = model(x, targets=y)
            loss = out["loss"] / GRAD_ACCUM

        loss.backward()
        accum_loss += out["loss"].item() / GRAD_ACCUM
        accum_aux += out["aux_loss"].item() / GRAD_ACCUM

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()
    global_step += 1

    # Log
    if global_step % LOG_EVERY == 0:
        dt = time.time() - step_start
        lr = scheduler.get_last_lr()[0]
        print(f"step={global_step:06d} | loss={accum_loss:.4f} | aux={accum_aux:.4f} | "
              f"grad_norm={grad_norm:.3f} | lr={lr:.2e} | dt={dt:.2f}s")
        step_start = time.time()

    # Checkpoint
    if global_step % SAVE_EVERY == 0:
        ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
        save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
        print(f"Saved checkpoint: {ckpt_path}")
        # Keep only last 2 checkpoints
        existing = sorted(Path(CHECKPOINT_DIR).glob("*.pt"))
        while len(existing) > 2:
            old = existing.pop(0)
            old.unlink()
            print(f"Removed old checkpoint: {old}")

    # Eval
    if global_step % EVAL_EVERY == 0:
        eval_cfg = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
        evaluator = Evaluator(model=model, eval_config=eval_cfg, device=device)
        metrics = evaluator.evaluate(eval_dataset)
        print(f"[eval] step={global_step:06d} | {metrics}")
        model.train()

# Final save
ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
print(f"Training complete! Final checkpoint: {ckpt_path}")

## 5. Evaluate the trained model

In [ ]:
from arman.training import Evaluator, EvalConfig

eval_config = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
evaluator = Evaluator(model=model, eval_config=eval_config, device=device)

metrics = evaluator.evaluate(eval_dataset)
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  Loss:           {metrics.loss:.4f}")
print(f"  Perplexity:     {metrics.perplexity:.2f}")
print(f"  Top-1 Accuracy: {metrics.top1_accuracy*100:.2f}%")
print(f"  Top-5 Accuracy: {metrics.top5_accuracy*100:.2f}%")
print(f"  MRR:            {metrics.mrr:.4f}")
print("=" * 50)

## 6. Generate text from the trained model

In [ ]:
from transformers import AutoTokenizer
from generate import generate

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Encode a prompt
prompt = "The most important concept in machine learning is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Generate
model.eval()
output_ids = generate(
    model,
    input_ids,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
)

# Decode
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

## 7. Export to HuggingFace Hub

In [ ]:
# ============================================================
# Export ArmanNN to HuggingFace-compatible format & push to Hub
# ============================================================

import json
import os
from pathlib import Path
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import HfApi, create_repo

# --- Config ---
HF_REPO_ID = "mhd-rahman/arman-nn"  # Change to your HF username/repo
EXPORT_DIR = "hf_export"
CHECKPOINT_PATH = None  # Set to a specific checkpoint, or None = use latest

# Find latest checkpoint if not specified
if CHECKPOINT_PATH is None:
    from arman.training.checkpointing import find_latest_checkpoint
    CHECKPOINT_PATH = find_latest_checkpoint("checkpoints")
    print(f"Using latest checkpoint: {CHECKPOINT_PATH}")

# Load checkpoint
state = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model_state = state["model"]
config_dict = state["config"]
step = state["step"]
print(f"Loaded checkpoint at step {step}")

# Create export directory
export_path = Path(EXPORT_DIR)
export_path.mkdir(parents=True, exist_ok=True)

# 1. Save config.json (HF-style)
hf_config = {
    "architectures": ["ArmanNN"],
    "model_type": "arman-nn",
    "auto_map": {
        "AutoConfig": "configuration_arman--ArmanConfig",
        "AutoModelForCausalLM": "modeling_arman--ArmanForCausalLM",
    },
    **config_dict,
    "torch_dtype": "bfloat16",
    "training_step": step,
}
with open(export_path / "config.json", "w") as f:
    json.dump(hf_config, f, indent=2)
print("Saved config.json")

# 2. Save model weights as safetensors
save_safetensors(model_state, str(export_path / "model.safetensors"))
print(f"Saved model.safetensors ({sum(p.numel() for p in model_state.values()):,} parameters)")

# 3. Save generation_config.json
gen_config = {
    "max_new_tokens": 256,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "repetition_penalty": 1.1,
    "do_sample": True,
}
with open(export_path / "generation_config.json", "w") as f:
    json.dump(gen_config, f, indent=2)
print("Saved generation_config.json")

# 4. Create configuration_arman.py (custom config for AutoConfig)
config_code = '''"""ArmanNN configuration for HuggingFace transformers."""
from transformers import PretrainedConfig

class ArmanConfig(PretrainedConfig):
    model_type = "arman-nn"

    def __init__(
        self,
        vocab_size=50257,
        d_model=1024,
        n_layers=12,
        n_heads=16,
        max_seq_len=1024,
        dropout=0.0,
        ssm_state_size=128,
        ssm_kernel_size=5,
        mlp_hidden=2816,
        n_experts=4,
        moe_top_k=2,
        expert_hidden=1792,
        graph_layers=2,
        memory_slots=128,
        memory_top_k=4,
        use_attention=True,
        use_ssm=True,
        use_mlp=True,
        use_moe=True,
        use_graph=False,
        use_memory=False,
        use_router=True,
        tie_embeddings=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.ssm_state_size = ssm_state_size
        self.ssm_kernel_size = ssm_kernel_size
        self.mlp_hidden = mlp_hidden
        self.n_experts = n_experts
        self.moe_top_k = moe_top_k
        self.expert_hidden = expert_hidden
        self.graph_layers = graph_layers
        self.memory_slots = memory_slots
        self.memory_top_k = memory_top_k
        self.use_attention = use_attention
        self.use_ssm = use_ssm
        self.use_mlp = use_mlp
        self.use_moe = use_moe
        self.use_graph = use_graph
        self.use_memory = use_memory
        self.use_router = use_router
        self.tie_embeddings = tie_embeddings
'''
with open(export_path / "configuration_arman.py", "w") as f:
    f.write(config_code)
print("Saved configuration_arman.py")

# 5. Create modeling_arman.py (custom model for AutoModelForCausalLM)
modeling_code = '''"""ArmanNN model for HuggingFace transformers."""
import sys
import os
from pathlib import Path

import torch
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from .configuration_arman import ArmanConfig as HFArmanConfig

# Import the actual model code
sys.path.insert(0, str(Path(__file__).parent))
from arman.model.config import ArmanConfig as NativeConfig
from arman.model.model import ArmanNN


class ArmanForCausalLM(PreTrainedModel):
    config_class = HFArmanConfig
    supports_gradient_checkpointing = True

    def __init__(self, config: HFArmanConfig):
        super().__init__(config)
        # Convert HF config to native config
        native_config = NativeConfig(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            n_layers=config.n_layers,
            n_heads=config.n_heads,
            max_seq_len=config.max_seq_len,
            dropout=config.dropout,
            ssm_state_size=config.ssm_state_size,
            mlp_hidden=config.mlp_hidden,
            n_experts=config.n_experts,
            moe_top_k=config.moe_top_k,
            expert_hidden=config.expert_hidden,
            graph_layers=config.graph_layers,
            memory_slots=config.memory_slots,
            memory_top_k=config.memory_top_k,
            use_attention=config.use_attention,
            use_ssm=config.use_ssm,
            use_mlp=config.use_mlp,
            use_moe=config.use_moe,
            use_graph=config.use_graph,
            use_memory=config.use_memory,
            use_router=config.use_router,
            tie_embeddings=config.tie_embeddings,
        )
        self.model = ArmanNN(native_config)

    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, **kwargs):
        out = self.model(input_ids, targets=labels, past_key_values=past_key_values)
        return CausalLMOutputWithPast(
            loss=out["loss"],
            logits=out["logits"],
            past_key_values=out["present_key_values"],
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, **kwargs):
        if past_key_values is not None:
            input_ids = input_ids[:, -1:]
        return {"input_ids": input_ids, "past_key_values": past_key_values}

    def _set_gradient_checkpointing(self, module, value=False):
        if value:
            self.model.enable_gradient_checkpointing()
        else:
            self.model.disable_gradient_checkpointing()
'''
with open(export_path / "modeling_arman.py", "w") as f:
    f.write(modeling_code)
print("Saved modeling_arman.py")

# 6. Copy the arman/ source package (needed for the model code)
import shutil
if (export_path / "arman").exists():
    shutil.rmtree(export_path / "arman")
shutil.copytree("arman", export_path / "arman")
print("Copied arman/ package")

# 7. Copy tokenizer files
tokenizer.save_pretrained(str(export_path))
print("Saved tokenizer")

# 8. Create model card
model_card = f'''---
language: en
license: apache-2.0
tags:
  - arman-nn
  - hybrid-architecture
  - attention
  - ssm
  - moe
  - causal-lm
library_name: transformers
pipeline_tag: text-generation
---

# ArmanNN

A hybrid language model combining Causal Attention, Selective SSM (parallel scan),
Sparse Mixture-of-Experts, with learned fusion gates and path routers.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{HF_REPO_ID}", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")

inputs = tokenizer("The future of AI is", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Architecture

- **Parameters:** {sum(p.numel() for p in model_state.values()):,}
- **Layers:** {config_dict.get("n_layers", 12)}
- **d_model:** {config_dict.get("d_model", 1024)}
- **Heads:** {config_dict.get("n_heads", 16)}
- **Experts:** {config_dict.get("n_experts", 4)} (top-{config_dict.get("moe_top_k", 2)})
- **Max seq length:** {config_dict.get("max_seq_len", 1024)}
- **Trained steps:** {step:,}

## Training Data

Mixed pretraining corpus:
- FineWeb-Edu (62.5%)
- Code - StarCoder (12.5%)
- Wikipedia (8.3%)
- OpenWebMath (16.7%)
'''
with open(export_path / "README.md", "w") as f:
    f.write(model_card)
print("Saved README.md (model card)")

print(f"\n{'='*50}")
print(f"Export complete! Files in: {EXPORT_DIR}/")
print(f"{'='*50}")

# 9. Push to HuggingFace Hub
PUSH_TO_HUB = True  # Set to False if you just want to export locally

if PUSH_TO_HUB:
    print(f"\nPushing to HuggingFace Hub: {HF_REPO_ID}")
    api = HfApi()
    create_repo(HF_REPO_ID, exist_ok=True)
    api.upload_folder(
        folder_path=EXPORT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"Pushed! Model available at: https://huggingface.co/{HF_REPO_ID}")
    print(f"\nLoad with:")
    print(f"  model = AutoModelForCausalLM.from_pretrained(\"{HF_REPO_ID}\", trust_remote_code=True)")
    print(f"  tokenizer = AutoTokenizer.from_pretrained(\"{HF_REPO_ID}\")")